# 02. Modern extraction: INE/PORDATA and CFP

Load the continuous modern B.9 bridge, independently parse the CFP ESA 2010 account and debt workbooks, and compare the two modern sources against each other before either is used downstream.

**Reads**

- `data/raw/pordata/pordata_2785_balance_by_level_1995_2025.csv`
- `data/raw/cfp/cfp_sec2010_annual_general_government_2026-04-15.xlsx`
- `data/raw/cfp/cfp_sec2010_annual_subsectors_2026-04-15.xlsx`
- `data/raw/cfp/cfp_rel_04_2026_social_security_underlying_data.xlsx`

**Writes**

- Nothing. The pipeline persists these extractions; here they are inspected.

**Method reference:** `METHODOLOGY.md` sections 3-4 and 10

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. Two independent modern sources

The modern segment uses one source as the continuous balance bridge and a second,
independently parsed source for detailed components:

- **INE via PORDATA** provides the four B.9 balances for 1995-2025 and is the
  canonical modern bridge;
- **CFP ESA 2010 workbooks** provide revenue, expenditure, interest, investment,
  Maastricht debt and stock-flow adjustments, and act as a cross-check on the
  bridge.

In [ ]:
from portugal_fiscal_balance.sources.cfp import (
    extract_cfp_annual,
    extract_social_security_detail,
)
from portugal_fiscal_balance.sources.pordata import load_balance_snapshot

modern = load_balance_snapshot(RAW / 'pordata' / 'pordata_2785_balance_by_level_1995_2025.csv')
cfp = extract_cfp_annual(
    RAW / 'cfp' / 'cfp_sec2010_annual_general_government_2026-04-15.xlsx',
    RAW / 'cfp' / 'cfp_sec2010_annual_subsectors_2026-04-15.xlsx',
)
ss_systems, ss_detail = extract_social_security_detail(
    RAW / 'cfp' / 'cfp_rel_04_2026_social_security_underlying_data.xlsx'
)
print('pordata balance bridge:', modern.shape)
print('cfp accounts:', cfp.accounts.shape)
print('cfp debt and stock-flow:', cfp.debt.shape)

In [ ]:
display(modern.tail(8))

## 2. Coverage of the CFP account panel

General Government components start in 1995; the three subsectors start in 2000.
That asymmetry is a property of the source and is preserved rather than filled.

In [ ]:
coverage = cfp.accounts.groupby('sector')['year'].agg(['min', 'max', 'count'])
display(coverage)

## 3. Do the two modern sources agree?

The PORDATA bridge is published rounded, and the CFP workbook is a separate
compilation, so exact equality is not expected. What matters is that the
differences stay at rounding scale instead of revealing a parsing error.

In [ ]:
from portugal_fiscal_balance.processing.validation import compare_modern_balance_sources

comparison = compare_modern_balance_sources(modern, cfp.general_government, cfp.subsectors)
difference_columns = [column for column in comparison.columns if column.endswith('_source_difference_m_eur')]
summary = (
    comparison[difference_columns]
    .abs()
    .max()
    .rename('max_abs_difference_m_eur')
    .to_frame()
    .round(1)
)
display(summary)
display(comparison[['year', *difference_columns]].tail(8).round(1))

In [ ]:
figure = figures.modern_source_differences(comparison)

## 4. Social Security budget tables

The CFP Social Security report is a **different accounting boundary** from the
ESA 2010 Social Security Funds sector. It is extracted here so that notebook 09
can analyse the internal Previdential / Citizenship / Special Regimes split
without merging it into the national-accounts balance.

In [ ]:
display(ss_systems)
display(ss_detail.set_index('year').T)

## Interpretation limits

1. The PORDATA bridge is **rounded at source**. It is used for the balance panel,
   not for component-level arithmetic.
2. CFP figures for the most recent year are **provisional** in the same sense as
   the official statistics they compile.
3. Social Security budget-system tables and ESA 2010 Social Security Funds
   accounts are **never combined into one series**.

---

[Previous: 01. Historical extraction: Banco de Portugal / INE](01_extract_historical_data.ipynb) | [Next: 03. Harmonisation and validation](03_harmonize_and_validate.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```